In [ ]:
IN_COLAB = 'google.colab' in str(get_ipython())
print(f'IN_COLAB={IN_COLAB}')


### Environment detection
This sets a shared Colab/local flag and keeps notebook behavior consistent across development and training environments.

### Configure project paths
In Colab we mount Drive and keep all checkpoints persistent. Locally we use the repository path.

In [ ]:
import sys
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/piano_model')
    BASE_PATH.mkdir(parents=True, exist_ok=True)
    PROJECT_ROOT = BASE_PATH / 'piano_midi_model'
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
    BASE_PATH = PROJECT_ROOT

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT={PROJECT_ROOT}')


### Install dependencies
This installs Colab dependencies (including CUDA Mamba) in Colab and CPU-safe dependencies locally.

In [ ]:
import subprocess
import sys

req_file = PROJECT_ROOT / ('requirements_colab.txt' if IN_COLAB else 'requirements.txt')
if req_file.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req_file)], check=False)
else:
    print(f'Requirements file not found: {req_file}')


### Load config and instantiate Mamba-only model
This is the ablation run with `use_cfc=False` to isolate the value of Mamba-style sequence modeling alone.

In [ ]:
from config import DataConfig, ModelConfig, TrainConfig
from model.hybrid import PianoHybridModel

data_cfg = DataConfig()
model_cfg = ModelConfig(use_cfc=False, use_mamba=True)
train_cfg = TrainConfig(max_epochs=50, batch_size=8, grad_accumulation_steps=4, device='auto')

if IN_COLAB:
    data_cfg.maestro_path = str(BASE_PATH / 'maestro-v3.0.0')
    data_cfg.tokenizer_path = str(BASE_PATH / 'tokenizer.json')
    data_cfg.processed_path = str(BASE_PATH / 'processed')
    train_cfg.checkpoint_dir = str(BASE_PATH / 'checkpoints_mamba_only')

model_cfg.vocab_size = data_cfg.vocab_size
model_cfg.max_sequence_length = data_cfg.max_sequence_length
mamba_model = PianoHybridModel(model_cfg)
mamba_model


### Log model summary
This confirms parameter counts and whether true Mamba kernels or GRU fallback are active.

In [ ]:
from utils.logging_utils import log_model_summary

log_model_summary(mamba_model, model_cfg)


### Create DataLoaders
We reuse the same preprocessed token dataset and split strategy for fair comparison with baseline and hybrid runs.

In [ ]:
from data.dataset import create_dataloaders

train_loader, val_loader, test_loader = create_dataloaders(data_cfg, train_cfg)
print('Train batches:', len(train_loader))
print('Val batches:', len(val_loader))
print('Test batches:', len(test_loader))


### Build trainer
The trainer adds warmup + cosine schedule, mixed precision on CUDA, and automatic best/latest checkpoints.

In [ ]:
from data.tokenizer import PianoTokenizer
from training.trainer import Trainer

tokenizer = PianoTokenizer.load(data_cfg.tokenizer_path)
trainer = Trainer(
    model=mamba_model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_cfg,
    data_config=data_cfg,
    tokenizer=tokenizer,
)
trainer


### Train Mamba-only model
This is the full run used to compare against both the baseline and the hybrid CfC+Mamba architecture.

In [ ]:
history_mamba = trainer.train()
history_mamba.keys()


### Compare Mamba-only vs baseline curves
This overlays the current run with baseline history to quickly assess whether the Mamba stack improves optimization.

In [ ]:
import matplotlib.pyplot as plt
import torch
from pathlib import Path

baseline_state_candidates = [
    Path(train_cfg.checkpoint_dir).parent / 'checkpoints_baseline' / 'best_state.pt',
    PROJECT_ROOT / 'checkpoints_baseline' / 'best_state.pt',
]
baseline_hist = None
for p in baseline_state_candidates:
    if p.exists():
        baseline_hist = torch.load(p, map_location='cpu').get('history', None)
        print(f'Loaded baseline history from {p}')
        break

plt.figure(figsize=(9, 4))
if baseline_hist is not None:
    plt.plot(baseline_hist.get('val_loss', []), label='baseline_val', linewidth=2)
plt.plot(history_mamba['val_loss'], label='mamba_only_val', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Baseline vs Mamba-only')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### Generate one continuation, visualize, and listen
This mirrors the baseline qualitative check so Mamba-only behavior can be compared directly in both pianoroll and audio.

In [ ]:
import shutil
import subprocess
from pathlib import Path

from IPython.display import Audio, display

from utils.midi_utils import compare_pianorolls

seed_batch, _ = next(iter(test_loader))
seed_tokens = seed_batch[0].tolist()
generated = mamba_model.generate(seed_tokens, max_new_tokens=data_cfg.continuation_length)
generated_continuation = generated[len(seed_tokens):]

out_dir = Path(train_cfg.checkpoint_dir) / 'qualitative'
out_dir.mkdir(parents=True, exist_ok=True)
seed_mid = out_dir / 'mamba_seed.mid'
gen_full_mid = out_dir / 'mamba_generated_full.mid'
gen_cont_mid = out_dir / 'mamba_generated_continuation.mid'
tokenizer.decode(seed_tokens, seed_mid)
tokenizer.decode(generated, gen_full_mid)
tokenizer.decode(generated_continuation, gen_cont_mid)
compare_pianorolls(seed_mid, gen_cont_mid)

wav_path = out_dir / 'mamba_generated.wav'
rendered = False
soundfont_candidates = ['/usr/share/sounds/sf2/FluidR3_GM.sf2', '/usr/share/soundfonts/default.sf2']
soundfont = next((p for p in soundfont_candidates if Path(p).exists()), None)
if IN_COLAB and shutil.which('fluidsynth') is None:
    subprocess.run(['apt-get', '-y', 'install', 'fluidsynth'], check=False)
if shutil.which('fluidsynth') and soundfont:
    cmd = ['fluidsynth', '-ni', soundfont, str(gen_full_mid), '-F', str(wav_path), '-r', '44100']
    subprocess.run(cmd, check=False)
    rendered = wav_path.exists()
if rendered:
    display(Audio(filename=str(wav_path), rate=44100))
else:
    print('Audio rendering skipped (fluidsynth/soundfont missing).')


### Save explicit final checkpoint
Trainer already saves best/latest checkpoints during training; this adds a tagged final snapshot for easier bookkeeping.

In [ ]:
final_val = history_mamba['val_loss'][-1] if history_mamba['val_loss'] else 0.0
trainer.save_checkpoint(epoch=train_cfg.max_epochs, val_loss=final_val, tag='mamba_only_final')
print(f'Checkpoint directory: {train_cfg.checkpoint_dir}')
